In [1]:
import yfinance as yf
import numpy as np
import pandas as pd

tickers = ['META', 'TSLA', 'NFLX', 'BA', 'T', 'MGM', 'PG', '^GSPC']
start_date = '2015-01-01'
end_date = '2025-01-01'
data = yf.download(tickers, start=start_date, end=end_date)

# Display the adjusted close prices
display(data['Close'])

/tmp/ipykernel_6450/808077539.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start=start_date, end=end_date)
[*********************100%***********************]  8 of 8 completed


Ticker,BA,META,MGM,NFLX,PG,T,TSLA,^GSPC
Date,,,,,,,,
2015-01-02,113.657211,77.839165,19.853611,4.984857,66.049400,11.403403,14.620667,2058.199951
2015-01-05,112.870064,76.588982,19.305794,4.731143,65.735374,11.295666,14.006000,2020.579956
2015-01-06,111.540619,75.557068,18.569075,4.650143,65.435913,11.312499,14.085333,2002.609985
2015-01-07,113.272354,75.557068,19.296349,4.674286,65.779198,11.378454,14.063333,2025.900024
2015-01-08,115.275276,77.571266,19.570259,4.778000,66.531395,11.491658,14.041333,2062.139893
...,...,...,...,...,...,...,...,...
2024-12-24,179.339996,605.321289,34.740002,93.211998,163.431107,21.486578,462.279999,6040.040039
2024-12-26,180.380005,600.938843,34.889999,92.414001,164.611313,21.495939,454.130005,6037.589844
2024-12-27,180.720001,597.413086,34.770000,90.754997,164.001877,21.402315,431.660004,5970.839844


In [2]:
# Calculate the maximum closing price for each asset
max_prices = data["Close"].max()

# Display the result
print("Maximum Price of Each Asset (2015-2025):\n")
display(max_prices)


Maximum Price of Each Asset (2015-2025):/n


,0
Ticker,
BA,430.299957
META,629.643799
MGM,50.900002
NFLX,93.655998
PG,173.840256
T,22.357273
TSLA,479.859985
^GSPC,6090.270020


In [3]:
# average closing price for each asset
average_prices = data['Close'].mean()

# results
print("Average Price of Each Asset (2015-2025):\n")
display(average_prices)

Average Price of Each Asset (2015-2025):



,0
Ticker,
BA,211.956333
META,220.613846
MGM,30.984598
NFLX,33.598515
PG,102.121926
T,15.096497
TSLA,115.679423
^GSPC,3356.124980


In [4]:
import plotly.express as px

# normalized data
normalized_data = (data['Close'] / data['Close'].iloc[0]) * 100

# To use Plotly Express easily, we convert our data to a 'long' format
df_plotly = normalized_data.reset_index()
df_melted = df_plotly.melt(id_vars='Date', var_name='Ticker', value_name='Normalized Price')

# Create the interactive line chart
fig = px.line(
    df_melted,
    x='Date',
    y='Normalized Price',
    color='Ticker',
    title='Evolution of the Asset prices Over Time',
)

# Add a unified hover template to compare all prices on a specific day easily
fig.update_layout(hovermode='x unified')

# Show the interactive plot
fig.show()

### Task 2: Exploratory Data Analysis (EDA) - Outperformers


In [5]:
# Get the final normalized prices (the last row of the dataset)
final_prices = normalized_data.iloc[-1]

# Get the final value for the S&P 500
gspc_final = final_prices['^GSPC']

# Filter for assets that have a higher final value than the S&P 500
outperformers = final_prices[final_prices > gspc_final]

# Display the results
print(f"S&P 500 Final Relative Growth: {gspc_final:.2f} (from a baseline of 100)\n")
print("Tickers that exceeded the S&P 500:")
display(outperformers.sort_values(ascending=False))

S&P 500 Final Relative Growth: 285.77 (from a baseline of 100)

Tickers that exceeded the S&P 500:


,2024-12-31
Ticker,
TSLA,2762.117376
NFLX,1788.055351
META,749.198900


In [6]:
# Calculate daily returns (percentage change from the previous day)
daily_returns = data['Close'].pct_change().dropna()

# Calculate the correlation matrix
correlation_matrix = daily_returns.corr()

# Isolate the correlation with the S&P 500 (^GSPC) and remove the S&P 500 itself (which would be 1.0)
sp500_correlation = correlation_matrix['^GSPC'].drop('^GSPC').sort_values(ascending=False)

# Display the results
print("Correlation of Daily Returns with the S&P 500:\n")
display(sp500_correlation)

Correlation of Daily Returns with the S&P 500:



,^GSPC
Ticker,
MGM,0.622865
META,0.605774
BA,0.598240
PG,0.533311
T,0.487152
NFLX,0.477190
TSLA,0.465115


Based on this information, we notice that the stocks have a positive correlation meaning that they all moved in the same direction as the S&P 500. More especifically, we can take notice that MGM is the most closely tied to the S&P 500 while TSLA is the least.

### Task 2: Exploratory Data Analysis (EDA) - Outliers & Unusual Periods (COVID-19)


In [7]:
import plotly.express as px

# Isolate the data for the year 2020
covid_period = data['Close'].loc['2020-01-01':'2020-12-31']

# Normalize the prices so everything starts at 100 on Jan 1, 2020
covid_normalized = (covid_period / covid_period.iloc[0]) * 100

# Prepare data for Plotly Express
df_covid_melted = covid_normalized.reset_index().melt(id_vars='Date', var_name='Ticker', value_name='Normalized Price')

# Create the interactive plot
fig = px.line(
    df_covid_melted,
    x='Date',
    y='Normalized Price',
    color='Ticker',
    title='Impact of COVID-19 on Asset Prices (2020)'
)

# Highlight the approximate crash period (Feb 19 to Mar 23)
fig.add_vrect(
    x0='2020-02-19', x1='2020-03-23',
    fillcolor='red', opacity=0.2,
    layer='below', line_width=0,
    annotation_text='COVID-19 Crash', annotation_position='top left'
)

fig.update_layout(
    hovermode='x unified',
    xaxis_title='Date',
    yaxis_title='Normalized Price (Jan 1, 2020 = 100)'
)
fig.show()

# Calculate the maximum drawdown (percentage drop) during the crash window
crash_window = covid_period.loc['2020-02-19':'2020-03-23']
peak_prices = crash_window.max()
trough_prices = crash_window.min()

# Calculate percentage drop
drawdowns = ((trough_prices - peak_prices) / peak_prices) * 100

print("Maximum Percentage Drop during COVID-19 Crash (Feb 19 - Mar 23, 2020):\n")
display(drawdowns.sort_values())

Maximum Percentage Drop during COVID-19 Crash (Feb 19 - Mar 23, 2020):



,0
Ticker,
MGM,-77.758455
BA,-71.915458
TSLA,-60.626539
^GSPC,-33.924960
META,-32.865885
T,-30.665631
PG,-22.888728
NFLX,-22.618397


Based on the graph and table provided, we notice that during this time period of COVID-19, all of these stocks, even the S&P 500, experienced a massive crash in price. However, with the inclusion of the data table, the information being depicted shows the total value percentage each stock, including the S&P 500, lost during that specific window. With that being said, we see the data table highlighting MGM, BA, and TSLA being the top 3 companies that experienced a severe crash in stock price during this time period in regards to the specific companies we are currently looking at.

### Task 3: Daily Returns Function
Develop a function to calculate the daily returns for all assets in the dataset.

In [8]:
def calculate_daily_returns(price_df):
    """
    Calculates the daily returns (percentage change) for a given DataFrame of prices.

    Args:
        price_df (pd.DataFrame): DataFrame containing asset prices.

    Returns:
        pd.DataFrame: DataFrame containing daily returns, with the first NaN row removed.
    """
    # Use pct_change() to calculate daily returns and dropna() to remove the first row of NaNs
    returns = price_df.pct_change().dropna()
    return returns

# Apply the function to our closing prices
daily_returns = calculate_daily_returns(data['Close'])

# Display the first few rows of the daily returns
print("Daily Returns (First 5 Rows):\n")
display(daily_returns.head())

Daily Returns (First 5 Rows):



Ticker,BA,META,MGM,NFLX,PG,T,TSLA,^GSPC
Date,,,,,,,,
2015-01-05,-0.006926,-0.016061,-0.027593,-0.050897,-0.004754,-0.009448,-0.042041,-0.018278
2015-01-06,-0.011779,-0.013473,-0.038161,-0.017121,-0.004556,0.001490,0.005664,-0.008893
2015-01-07,0.015526,0.000000,0.039166,0.005192,0.005246,0.005830,-0.001562,0.011630
2015-01-08,0.017682,0.026658,0.014195,0.022188,0.011435,0.009949,-0.001564,0.017888
2015-01-09,-0.001973,-0.005628,-0.010135,-0.015458,-0.009330,-0.002985,-0.018802,-0.008404


### Task 3.1: Risk and Return Summary


In [9]:
import numpy as np

# Assuming 252 trading days in a year
trading_days = 252

# Calculate Annualized Return (Mean daily return * 252)
annualized_return = daily_returns.mean() * trading_days

# Calculate Annualized Volatility (Standard deviation of daily returns * sqrt(252))
annualized_volatility = daily_returns.std() * np.sqrt(trading_days)

# Create a summary DataFrame
summary_stats = pd.DataFrame({
    'Annualized Return (Reward)': annualized_return,
    'Annualized Volatility (Risk)': annualized_volatility
})

# Format the output as percentages for better readability
formatted_summary = summary_stats.map(lambda x: f"{x*100:.2f}%")

print("Annualized Risk and Return Summary (2015-2025):\n")
display(formatted_summary.sort_values(by='Annualized Return (Reward)', ascending=False))

Annualized Risk and Return Summary (2015-2025):



,Annualized Return (Reward),Annualized Volatility (Risk)
Ticker,,
T,8.85%,22.71%
TSLA,49.54%,57.16%
NFLX,38.56%,43.63%
META,27.29%,37.49%
MGM,15.88%,45.16%
BA,12.57%,40.29%
^GSPC,12.12%,17.83%
PG,10.71%,18.47%


### Task 3.2: Time-Series Plots of Returns


In [10]:
import plotly.express as px

# Prepare the daily returns data for Plotly by converting it to a 'long' format
df_returns_melted = daily_returns.reset_index().melt(id_vars='Date', var_name='Ticker', value_name='Daily Return')

# Create the interactive time-series plot
fig = px.line(
    df_returns_melted,
    x='Date',
    y='Daily Return',
    color='Ticker',
    title='Time-Series of Daily Returns (2015-2025)'
)

# Format the layout for better readability
fig.update_layout(
    yaxis_tickformat='.2%',  # Display y-axis as percentages
    hovermode='x unified',   # Show all values for a given day on hover
    xaxis_title='Date',
    yaxis_title='Daily Return'
)

# Show the interactive plot
fig.show()

### Task 3.3: Histograms of Daily Returns


In [11]:
import plotly.express as px

# We reuse the df_returns_melted dataframe created in the previous step
# Create a faceted histogram (a separate plot for each ticker)
fig = px.histogram(
    df_returns_melted,
    x='Daily Return',
    facet_col='Ticker',
    facet_col_wrap=4,  # Create a 2x4 grid of plots
    nbins=100,         # Number of bins to group the returns into
    title='Distribution of Daily Returns by Asset (2015-2025)'
)

# Format the axes for better readability
fig.update_layout(showlegend=False, hovermode='x unified')
fig.update_xaxes(tickformat='.1%', title_text='')
fig.update_yaxes(title_text='Frequency')

# Show the interactive plot
fig.show()

Compare the volatility of the different stocks. Which appear most or least volatile? Answer below

Based off the shape of each stock's bell curve, we can take notice that the most volatile was TSLA. Tesla's bell curve is the widest showing its daily returns were spread out over a much larger range of percentages suggesting frequent price swings. As fro the one being the least volatile, it was PG, Proctor & Gamble. PG's bell curve was the tallest and narrowest suggesting its daily returns were tightly crowding around zero showing that extreme price swings are rare and the stock is highly stable.

In [12]:
import plotly.express as px

# 1. Correlation of Stock Prices
price_correlation = data['Close'].corr()

# Plot heatmap for prices
fig_prices = px.imshow(
    price_correlation,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Correlation Heatmap of Stock Prices (2015-2025)'
)
fig_prices.show()

# 2. Correlation of Daily Returns
returns_correlation = daily_returns.corr()

# Plot heatmap for returns
fig_returns = px.imshow(
    returns_correlation,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Correlation Heatmap of Daily Returns (2015-2025)'
)
fig_returns.show()

Based off the "Correlation Heatmap of Stock Prices", we can observe that most of the stocks, excluding AT&T's stock "T", have a strong correlation with the S&P 500 or GSPC. This shows that these companies stock prices grew with the S&P 500. Another pattern we can take notice to is Boeing's, or "BA", low or negative correlation with the other assets. The cause of this may have had to do with external troubles, such as COVID-19. One final pattern we can observe is that two tech companies, Netflix and Meta, share a high correlation, suggesting that within the last decade, sector-specific trends caused big tech companies to move together.

As for the second graph, "Correlation Heatmap of Daily Returns", one pattern is that there seems to be less correlation between each asset's daily returns compared to the stocks prices. This means that their daily price movements were less synchronized as they reacted to various events. Another pattern is that although the correlations are not as high as depicted in the first graph, we can still observe that the assets' daily returns still have a positive correlation with the S&P 500. With that being said, the S&P 500 still clearly has a market pull effect on the assets. A final pattern is the observation of diversification. When looking at Netflix and Proctor & Gamble or AT&T and Netflix, there is a very low correlation. That shows that those two companies operate in different sectors, therefore allowing the use of diversification in one's portfolio.


### Task 3.4: Simple Returns vs. Log Returns


In [13]:
# Calculate Log Returns using the natural logarithm of (Price today / Price yesterday)
log_returns = np.log(data['Close'] / data['Close'].shift(1)).dropna()

# Create a side-by-side comparison for the S&P 500 (^GSPC)
comparison_df = pd.DataFrame({
    'Simple Returns': daily_returns['^GSPC'],
    'Log Returns': log_returns['^GSPC']
})

# Calculate the mathematical difference
comparison_df['Difference'] = comparison_df['Simple Returns'] - comparison_df['Log Returns']

print("Comparison of Simple vs. Log Returns for the S&P 500 (First 5 Days):\n")
display(comparison_df.head())

print("\nSummary Statistics of the Difference (Simple - Log):\n")
display(comparison_df['Difference'].describe())


Comparison of Simple vs. Log Returns for the S&P 500 (First 5 Days):



,Simple Returns,Log Returns,Difference
Date,,,
2015-01-05,-0.018278,-0.018447,0.000169
2015-01-06,-0.008893,-0.008933,0.000040
2015-01-07,0.011630,0.011563,0.000067
2015-01-08,0.017888,0.017730,0.000158
2015-01-09,-0.008404,-0.008439,0.000036



Summary Statistics of the Difference (Simple - Log):



,Difference
count,2515.000000
mean,0.000063
std,0.000262
min,0.000000
25%,0.000002
50%,0.000012
75%,0.000050
max,0.007812


The difference between Simple Returns and Log Returns is that Simple Returns are asset additive. This means that a portfolio's simple return is simply just the weighted average of each individual simple returns. You must also multiply 1 + return over time to compound them. Simple Returns are also asymmetric, meaning that if a stock drops 50%, it needs a 100% simple return to get back to where it started.
On the other hand, Log returns are time additive. This means that if you want to find the total return over a 5 day week, you simply add the 5 daily log returns together. Log Returns are also symmetric, meaning that if a stock goes up 50% in log returns and then down 50%, its price is back to where it started at.   

### Part 4: Beta Calculation (Individual Securities)

**Beta = Covariance(Stock, Market) / Variance(Market)**

In [14]:
# Isolate the market returns (S&P 500)
market_returns = daily_returns['^GSPC']

# Calculate the variance of the market
market_variance = market_returns.var()

# Calculate the covariance of each stock with the market S&P 500
covariance_matrix = daily_returns.cov()
market_covariance = covariance_matrix['^GSPC']

# Calculate Beta: Covariance(Stock, Market) / Variance(Market)
betas = market_covariance / market_variance

# Remove the S&P 500 from the results (its beta is exactly 1.0) and sort
stock_betas = betas.drop('^GSPC').sort_values(ascending=False)

# Convert to a DataFrame for a clean display
beta_df = stock_betas.to_frame(name='Beta')
beta_df.index.name = 'Ticker'

print("Estimated Beta for Each Stock (Relative to S&P 500):\n")
display(beta_df)


Estimated Beta for Each Stock (Relative to S&P 500):



,Beta
Ticker,
MGM,1.577881
TSLA,1.491326
BA,1.351848
META,1.273905
NFLX,1.167859
T,0.620430
PG,0.552558


A beta greater than 1 implies that the security is more volatile than the market.

A beta less than 1 implies that the security is less volatile than the market usually adding more stability during market downturns.

A beta equal to 1 means the security moves with the market carrying the same systemic risk.

A beta less than 0 means the security moves the opposite direction of the market.

### Task 4.1: Beta Stability (Sub-Period Analysis)


In [15]:
# Define the two sub-periods
period_1 = daily_returns.loc['2015-01-01':'2019-12-31']
period_2 = daily_returns.loc['2020-01-01':'2024-12-31']

# Helper function to calculate beta for a given dataframe
def calculate_betas(returns_df):
    market_var = returns_df['^GSPC'].var()
    cov_matrix = returns_df.cov()
    market_cov = cov_matrix['^GSPC']
    betas = market_cov / market_var
    return betas.drop('^GSPC')

# Calculate betas for each period
beta_p1 = calculate_betas(period_1)
beta_p2 = calculate_betas(period_2)

# Combine into a single DataFrame for easy comparison
beta_comparison = pd.DataFrame({
    'Beta (2015-2019)': beta_p1,
    'Beta (2020-2024)': beta_p2
})

# Calculate the absolute difference
beta_comparison['Difference'] = beta_comparison['Beta (2020-2024)'] - beta_comparison['Beta (2015-2019)']

print("Beta Comparison Across Sub-Periods:\n")
display(beta_comparison.sort_values(by='Beta (2015-2019)', ascending=False))


Beta Comparison Across Sub-Periods:



,Beta (2015-2019),Beta (2020-2024),Difference
Ticker,,,
NFLX,1.530672,1.024389,-0.506282
MGM,1.447751,1.629586,0.181835
TSLA,1.269805,1.578275,0.308470
META,1.186714,1.308429,0.121716
BA,1.133304,1.439050,0.305746
T,0.638160,0.613561,-0.024599
PG,0.565942,0.547302,-0.018640


In the data shown above, we can observe the betas of each security between the time periods of 2015-2019 and 2020-2024. As for the significant changes that took place there were several. The first was Netflix, and it experienced a -0.50 change. The other changes were found in TSLA and BA. Both securities experienced a spike by +0.30.

### Task 4.2: Beta via Linear Regression


In [16]:
# Calculate the average daily return of the market (S&P 500)
market_avg_return = daily_returns['^GSPC'].mean()


# BOEING (BA)
ba_avg_return = daily_returns['BA'].mean()
ba_beta = stock_betas['BA']
ba_alpha = ba_avg_return - (ba_beta * market_avg_return)
print(f"BA Alpha: {ba_alpha:.6f}")

# META (META)
meta_avg_return = daily_returns['META'].mean()
meta_beta = stock_betas['META']
meta_alpha = meta_avg_return - (meta_beta * market_avg_return)
print(f"META Alpha: {meta_alpha:.6f}")

# MGM RESORTS (MGM)
mgm_avg_return = daily_returns['MGM'].mean()
mgm_beta = stock_betas['MGM']
mgm_alpha = mgm_avg_return - (mgm_beta * market_avg_return)
print(f"MGM Alpha: {mgm_alpha:.6f}")

# NETFLIX (NFLX)
nflx_avg_return = daily_returns['NFLX'].mean()
nflx_beta = stock_betas['NFLX']
nflx_alpha = nflx_avg_return - (nflx_beta * market_avg_return)
print(f"NFLX Alpha: {nflx_alpha:.6f}")

# PROCTER & GAMBLE (PG)
pg_avg_return = daily_returns['PG'].mean()
pg_beta = stock_betas['PG']
pg_alpha = pg_avg_return - (pg_beta * market_avg_return)
print(f"PG Alpha: {pg_alpha:.6f}")

# AT&T (T)
t_avg_return = daily_returns['T'].mean()
t_beta = stock_betas['T']
t_alpha = t_avg_return - (t_beta * market_avg_return)
print(f"T Alpha: {t_alpha:.6f}")

# TESLA (TSLA)
tsla_avg_return = daily_returns['TSLA'].mean()
tsla_beta = stock_betas['TSLA']
tsla_alpha = tsla_avg_return - (tsla_beta * market_avg_return)
print(f"TSLA Alpha: {tsla_alpha:.6f}")


BA Alpha: -0.000151
META Alpha: 0.000470
MGM Alpha: -0.000129
NFLX Alpha: 0.000968
PG Alpha: 0.000159
T Alpha: 0.000053
TSLA Alpha: 0.001249


In [17]:
# Calculate R-squared for each stock (Correlation with S&P 500 squared)

# BOEING (BA)
ba_corr = daily_returns['BA'].corr(daily_returns['^GSPC'])
ba_r2 = ba_corr * ba_corr
print(f"Return(BA) = {ba_alpha:.6f} + {ba_beta:.4f} * Return(S&P 500) | R²: {ba_r2:.4f}")

# META (META)
meta_corr = daily_returns['META'].corr(daily_returns['^GSPC'])
meta_r2 = meta_corr * meta_corr
print(f"Return(META) = {meta_alpha:.6f} + {meta_beta:.4f} * Return(S&P 500) | R²: {meta_r2:.4f}")

# MGM RESORTS (MGM)
mgm_corr = daily_returns['MGM'].corr(daily_returns['^GSPC'])
mgm_r2 = mgm_corr * mgm_corr
print(f"Return(MGM) = {mgm_alpha:.6f} + {mgm_beta:.4f} * Return(S&P 500) | R²: {mgm_r2:.4f}")

# NETFLIX (NFLX)
nflx_corr = daily_returns['NFLX'].corr(daily_returns['^GSPC'])
nflx_r2 = nflx_corr * nflx_corr
print(f"Return(NFLX) = {nflx_alpha:.6f} + {nflx_beta:.4f} * Return(S&P 500) | R²: {nflx_r2:.4f}")

# PROCTER & GAMBLE (PG)
pg_corr = daily_returns['PG'].corr(daily_returns['^GSPC'])
pg_r2 = pg_corr * pg_corr
print(f"Return(PG) = {pg_alpha:.6f} + {pg_beta:.4f} * Return(S&P 500) | R²: {pg_r2:.4f}")

# AT&T (T)
t_corr = daily_returns['T'].corr(daily_returns['^GSPC'])
t_r2 = t_corr * t_corr
print(f"Return(T) = {t_alpha:.6f} + {t_beta:.4f} * Return(S&P 500) | R²: {t_r2:.4f}")

# TESLA (TSLA)
tsla_corr = daily_returns['TSLA'].corr(daily_returns['^GSPC'])
tsla_r2 = tsla_corr * tsla_corr
print(f"Return(TSLA) = {tsla_alpha:.6f} + {tsla_beta:.4f} * Return(S&P 500) | R²: {tsla_r2:.4f}")

Return(BA) = -0.000151 + 1.3518 * Return(S&P 500) | R²: 0.3579
Return(META) = 0.000470 + 1.2739 * Return(S&P 500) | R²: 0.3670
Return(MGM) = -0.000129 + 1.5779 * Return(S&P 500) | R²: 0.3880
Return(NFLX) = 0.000968 + 1.1679 * Return(S&P 500) | R²: 0.2277
Return(PG) = 0.000159 + 0.5526 * Return(S&P 500) | R²: 0.2844
Return(T) = 0.000053 + 0.6204 * Return(S&P 500) | R²: 0.2373
Return(TSLA) = 0.001249 + 1.4913 * Return(S&P 500) | R²: 0.2163


The regression equation explains how the stock's return is predicted by the market. It uses beta as the multiplier for the S&P 500. As for the constant, it represents the stock's independent of the market.

For R-squared, this tells us the percentage of the stock's price movements which can be explained by the movements of the S&P 500. For example, BA's R-squared is 0.3579, that means 35.79% of BA's volatility is driven by the S&P 500 while 64.21% is driven by company specific events.

### Task 5: Capital Asset Pricing Model (CAPM)


**Formula:**
`Expected Return = Risk-Free Rate + Beta * (Expected Market Return - Risk-Free Rate)`



In [18]:

# Download the 10-Year Treasury Yield (^TNX) to use as our risk-free rate
tnx_data = yf.download('^TNX', start=start_date, end=end_date)

# Get the very last closing yield
latest_yield = float(tnx_data['Close'].squeeze().iloc[-1])

# Convert it to a decimal (e.g. 4.5% -> 0.045)
risk_free_rate = latest_yield / 100

# Get the expected market return from our earlier calculations (S&P 500 annualized return)
market_return = annualized_return['^GSPC']

# Calculate the Market Risk Premium (Market Return - Risk Free Rate)
market_risk_premium = market_return - risk_free_rate

print(f"Dynamic Risk-Free Rate (10-Year Treasury): {latest_yield:.2f}%")
print(f"Expected Market Return: {market_return * 100:.2f}%\n")
print("Expected Return using CAPM:\n")

# BOEING (BA)
ba_beta = stock_betas['BA']
ba_expected_return = risk_free_rate + (ba_beta * market_risk_premium)
ba_percent = ba_expected_return * 100
print(f"BA Expected Return: {ba_percent:.2f}%")

# META (META)
meta_beta = stock_betas['META']
meta_expected_return = risk_free_rate + (meta_beta * market_risk_premium)
meta_percent = meta_expected_return * 100
print(f"META Expected Return: {meta_percent:.2f}%")

# MGM RESORTS (MGM)
mgm_beta = stock_betas['MGM']
mgm_expected_return = risk_free_rate + (mgm_beta * market_risk_premium)
mgm_percent = mgm_expected_return * 100
print(f"MGM Expected Return: {mgm_percent:.2f}%")

# NETFLIX (NFLX)
nflx_beta = stock_betas['NFLX']
nflx_expected_return = risk_free_rate + (nflx_beta * market_risk_premium)
nflx_percent = nflx_expected_return * 100
print(f"NFLX Expected Return: {nflx_percent:.2f}%")

# PROCTER & GAMBLE (PG)
pg_beta = stock_betas['PG']
pg_expected_return = risk_free_rate + (pg_beta * market_risk_premium)
pg_percent = pg_expected_return * 100
print(f"PG Expected Return: {pg_percent:.2f}%")

# AT&T (T)
t_beta = stock_betas['T']
t_expected_return = risk_free_rate + (t_beta * market_risk_premium)
t_percent = t_expected_return * 100
print(f"T Expected Return: {t_percent:.2f}%")

# TESLA (TSLA)
tsla_beta = stock_betas['TSLA']
tsla_expected_return = risk_free_rate + (tsla_beta * market_risk_premium)
tsla_percent = tsla_expected_return * 100
print(f"TSLA Expected Return: {tsla_percent:.2f}%")

/tmp/ipykernel_6450/1109763053.py:2: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed

Dynamic Risk-Free Rate (10-Year Treasury): 4.57%
Expected Market Return: 12.12%

Expected Return using CAPM:

BA Expected Return: 14.77%
META Expected Return: 14.19%
MGM Expected Return: 16.48%
NFLX Expected Return: 13.39%
PG Expected Return: 8.74%
T Expected Return: 9.25%
TSLA Expected Return: 15.83%


### Task 5.1: Actual vs. CAPM-Predicted Returns


In [19]:
#  Gather our individual CAPM expected returns into a pandas Series
capm_expected = {
    'BA': ba_expected_return,
    'META': meta_expected_return,
    'MGM': mgm_expected_return,
    'NFLX': nflx_expected_return,
    'PG': pg_expected_return,
    'T': t_expected_return,
    'TSLA': tsla_expected_return
}
capm_series = pd.Series(capm_expected)

# Create a DataFrame comparing the actual annualized return to the CAPM return
capm_comparison = pd.DataFrame({
    'Actual Annual Return': annualized_return.drop('^GSPC'),
    'CAPM Expected Return': capm_series
})

# Calculate the difference (Alpha representation)
capm_comparison['Difference (Actual - CAPM)'] = capm_comparison['Actual Annual Return'] - capm_comparison['CAPM Expected Return']

# Format the dataframe as percentages
formatted_capm = capm_comparison.map(lambda x: f"{x*100:.2f}%")

print("Comparison of Actual vs. CAPM-Predicted Returns:\n")
display(formatted_capm)

Comparison of Actual vs. CAPM-Predicted Returns:



,Actual Annual Return,CAPM Expected Return,Difference (Actual - CAPM)
BA,12.57%,14.77%,-2.21%
META,27.29%,14.19%,13.11%
MGM,15.88%,16.48%,-0.60%
NFLX,38.56%,13.39%,25.17%
PG,10.71%,8.74%,1.96%
T,8.85%,9.25%,-0.40%
TSLA,49.54%,15.83%,33.71%


Overvalued:
- BA
- MGM
- T

Undervalued:
- TSLA
- NFLX
- META
- PG

In [20]:
import plotly.graph_objects as go
import numpy as np

# 1. Prepare the SML line data
# We extend the line a bit past our highest beta for a better visual
max_beta = stock_betas.max() + 0.3
sml_betas = np.linspace(0, max_beta, 100)
# SML Formula: Expected Return = Risk-Free Rate + Beta * Market Risk Premium
sml_returns = risk_free_rate + sml_betas * market_risk_premium

# Initialize the Plotly figure
fig = go.Figure()

# Add the SML line trace
fig.add_trace(go.Scatter(
    x=sml_betas,
    y=sml_returns,
    mode='lines',
    name='Security Market Line (SML)',
    line=dict(color='blue', dash='dash')
))

# 2. Add the individual stock actual returns
stock_names = stock_betas.index
actual_returns_for_stocks = annualized_return.loc[stock_names]

fig.add_trace(go.Scatter(
    x=stock_betas,
    y=actual_returns_for_stocks,
    mode='markers+text',
    name='Individual Stocks (Actual Return)',
    text=stock_names,
    textposition='top center',
    marker=dict(size=12, color='red', line=dict(width=1, color='DarkSlateGrey'))
))

# 3. Add the Market Portfolio Point (S&P 500) for reference
fig.add_trace(go.Scatter(
    x=[1.0],
    y=[market_return],
    mode='markers+text',
    name='Market Portfolio (^GSPC)',
    text=['S&P 500'],
    textposition='bottom right',
    marker=dict(size=14, color='green', symbol='star', line=dict(width=1, color='DarkSlateGrey'))
))

# Add a marker for the Risk-Free Rate (Beta = 0)
fig.add_trace(go.Scatter(
    x=[0],
    y=[risk_free_rate],
    mode='markers+text',
    name='Risk-Free Rate',
    text=['Risk-Free Rate'],
    textposition='top right',
    marker=dict(size=10, color='purple', symbol='triangle-up')
))

# Format the layout
fig.update_layout(
    title='Security Market Line (SML) vs Actual Performance (2015-2025)',
    xaxis_title='Systematic Risk (Beta)',
    yaxis_title='Annualized Return',
    yaxis_tickformat='.2%',  # Format y-axis as percentages
    hovermode='closest',
    template='plotly_white',
    showlegend=True
)

# Display the plot
fig.show()

Discuss the limitations of CAPM in explaining real-world market behavior.

When it comes to explaining real-world market behavior, CAPM has its limitations. One limitation is that the CAPM model focuses on the idea that the beta drives return. Instead, it is found that looking at the size of the company and its value can lead to even more accurate predictions. Not only that, but CAPM also assumes perfectly efficient markets where there is no transaction costs or taxes. Though, in reality, things like those have an effect on returns.